###  Metadata builder to retrieve filter bottoms from filename

In [1]:
# Setup: Import modules and define paths
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path FIRST (before importing from functions)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Define folders
wiertsema_input_dir = repo_root / 'output_data' / 'wiertsema'
fugro_input_dir = repo_root / 'output_data' / 'fugro'

print('Setup complete!')
print(f'Repo root: {repo_root}')
print(f'Wiertsema input: {wiertsema_input_dir}')
print(f'Fugro input: {fugro_input_dir}')

Setup complete!
Repo root: d:\Users\jvanruitenbeek\data_validation
Wiertsema input: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
Fugro input: d:\Users\jvanruitenbeek\data_validation\output_data\fugro


In [2]:
import re
import pandas as pd
from pathlib import Path

# Dataset roots
wiertsema_input_dir = repo_root / 'output_data' / 'wiertsema'
fugro_input_dir = repo_root / 'output_data' / 'fugro'

# --- Extraction helpers ---
fugro_pattern = re.compile(
    r"_([+-]?\d+(?:\.\d+)?)_m_NAP_avg\.csv$", re.IGNORECASE
)

def extract_fugro_depth(name: str):
    """
    Extract Fugro bottom filter as the float directly before '_m_NAP_avg'.
    Returns float or None.
    """
    m = fugro_pattern.search(name)
    if not m:
        return None
    try:
        return float(m.group(1))
    except ValueError:
        return None

# Wiertsema examples:
wiertsema_pattern = re.compile(r"_F(-[0-9]+(?:\.[0-9]+)?)")

def extract_wiertsema_depth(name: str):
    """
    Extract Wiertsema filter bottom depth (in meters).
    """
    m = wiertsema_pattern.search(name)
    if not m:
        return None

    raw_val = m.group(1)
    raw_val_clean = raw_val.replace(",", ".")

    try:
        if "." in raw_val_clean:
            mid = float(raw_val_clean)
        else:
            mid = float(raw_val_clean) / 100.0

        bottom = mid - 0.5
        return bottom
    except ValueError:
        return None

# --- Folder scanner ---
def scan_folder(dataset_root: Path, kind: str):
    """
    Scan a dataset root recursively and return rows with filename + hmin_pb.
    Looks for files in <origin>/only_csv/*.csv.
    kind: 'fugro' or 'wiertsema'
    """
    rows = []
    for f in sorted(dataset_root.glob('*/only_csv/*.csv')):
        source_origin_stem = f.parent.parent.name

        if kind == "fugro":
            depth = extract_fugro_depth(f.name)
        elif kind == "wiertsema":
            depth = extract_wiertsema_depth(f.name)
        else:
            depth = None

        rows.append({
            "filename": f.name,
            "source_origin_stem": source_origin_stem,
            "x": "",
            "y": "",
            "hmax_pb": "",
            "hmin_pb": depth if depth is not None else "",
        })

    return rows

# --- Build combined dataframe and save ---
rows_fugro = scan_folder(fugro_input_dir, "fugro")
rows_wiertsema = scan_folder(wiertsema_input_dir, "wiertsema")

all_rows = rows_fugro + rows_wiertsema
df_out = pd.DataFrame(all_rows)

output_csv = repo_root / "output_data" / "object_data.csv"
df_out.to_csv(output_csv, index=False)

print("Saved:", output_csv)

Saved: d:\Users\jvanruitenbeek\data_validation\output_data\object_data.csv


In [3]:
df_out

,filename,source_origin_stem,x,y,hmax_pb,hmin_pb
0,NL-2412417-HWM_B09-PB1_m_NAP.csv,4423-241417_PB_HOORN_01-01-2023 00_00_00_02-03...,,,,
1,NL-2412417-HWM_B09-PB2_m_NAP.csv,4423-241417_PB_HOORN_01-01-2023 00_00_00_02-03...,,,,
2,NL-2412417-HWM_B12-PB1_m_NAP.csv,4423-241417_PB_HOORN_01-01-2023 00_00_00_02-03...,,,,
3,NL-2412417-HWM_B13-PB1_m_NAP.csv,4423-241417_PB_HOORN_01-01-2023 00_00_00_02-03...,,,,
4,NL-2412417-HWM_B13-PB2_m_NAP.csv,4423-241417_PB_HOORN_01-01-2023 00_00_00_02-03...,,,,
...,...,...,...,...,...,...
329,86349-1 MB042PB01 B_BE0263+75_BIKR_GMW_PB1_F-4...,peilbuisdata_alle_sensoren_07042026,,,,-4.68
330,86349-1 MB043PB01 B_BE0263+75_BIT_GMW_PB1_F-45...,peilbuisdata_alle_sensoren_07042026,,,,-5.02
331,86349-1 MB045PB01 B_BE0328+3_BUKR_GMW_PB1_F-34...,peilbuisdata_alle_sensoren_07042026,,,,-3.99
332,86349-1 MB047PB01 B_BE0328+2_BIT_GMW_PB1_F-651...,peilbuisdata_alle_sensoren_07042026,,,,-7.01
